<a href="https://colab.research.google.com/github/CheredlaJasvina/DSA0302-NLP/blob/main/DSA0302_A_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 2

In [2]:
import re

text = "Patient received 500mg Paracetamol and 250mg Amoxicillin."

greedy = re.findall(r".*mg", text)
non_greedy = re.findall(r".*?mg", text)

print("Greedy Match:", greedy)
print("Non-Greedy Match:", non_greedy)

Greedy Match: ['Patient received 500mg Paracetamol and 250mg']
Non-Greedy Match: ['Patient received 500mg', ' Paracetamol and 250mg']


In [3]:
import re

text = "fever cough cough"

pattern1 = r"fever|cough+"
pattern2 = r"(fever|cough)+"

print(re.findall(pattern1, text))
print(re.findall(pattern2, text))

['fever', 'cough', 'cough']
['fever', 'cough', 'cough']


Task 3

In [5]:
import re

text = """Patient has fever and BP is 120/80 mmHg.
Given 500mg Paracetamol 2-times daily.
Hb level is 12.5 g/dL."""

pattern = r'\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?(?:mg/dL|g/dL|mmHg|mg|mcg|ml|°C)?|[A-Za-z]+(?:[.-][A-Za-z]+)*|[^\w\s]'

tokens = re.findall(pattern, text)

print("Tokens:")
print(tokens)

Tokens:
['Patient', 'has', 'fever', 'and', 'BP', 'is', '120/80', 'mmHg', '.', 'Given', '500mg', 'Paracetamol', '2', '-', 'times', 'daily', '.', 'Hb', 'level', 'is', '12.5', 'g', '/', 'dL', '.']


In [6]:
import re

text = """PATIENT has BP 120/80 mmHg.
Given 500mg Paracetamol BD on 05/11/2026."""

# Convert text to lowercase
text = text.lower()

# Expand abbreviations
rules = {
    "bp": "blood pressure",
    "bd": "twice daily",
    "od": "once daily",
    "tds": "three times daily",
    "qid": "four times daily",
    "hr": "heart rate",
    "htn": "hypertension",
    "dm": "diabetes mellitus",
    "iv": "intravenous",
    "po": "by mouth"
}

for old, new in rules.items():
    text = re.sub(r'\b' + old + r'\b', new, text)

# Standardise date
text = re.sub(
    r'(\d{2})/(\d{2})/(\d{4})',
    r'\3-\2-\1',
    text
)

# Standardise dosage spacing
text = re.sub(r'(\d+)\s*(mg|mcg|g|ml)\b',
              r'\1 \2', text)

print(text)

patient has blood pressure 120/80 mmhg.
given 500 mg paracetamol twice daily on 2026-11-05.


Task 4

In [7]:
import re

def validate_patient_id(patient_id):
    pattern = r'^MRN-?[0-9]{6}$'

    if re.fullmatch(pattern, patient_id):
        return "Valid Patient ID"
    else:
        return "Invalid Patient ID"

print(validate_patient_id("MRN123456"))
print(validate_patient_id("MRN-123456"))
print(validate_patient_id("MRN12345"))
print(validate_patient_id("MRN--123456"))

Valid Patient ID
Valid Patient ID
Invalid Patient ID
Invalid Patient ID


Task 7

In [8]:
lexicon = {
    "fractures": "fracture +N +PL",
    "patients": "patient +N +PL",
    "treated": "treat +V +PAST",
    "infecting": "infect +V +PROG",
    "diagnoses": "diagnose +V +3SG"
}

def morphological_parser(word):
    if word in lexicon:
        return lexicon[word]
    else:
        return "Word not found in lexicon"

words = [
    "fractures",
    "patients",
    "treated",
    "infecting",
    "diagnoses"
]

for word in words:
    print(word, "->", morphological_parser(word))

fractures -> fracture +N +PL
patients -> patient +N +PL
treated -> treat +V +PAST
infecting -> infect +V +PROG
diagnoses -> diagnose +V +3SG


Task 8

In [9]:
def apply_rules(word, suffix):

    # Rule 1: E deletion
    if word.endswith("e") and suffix == "ing":
        word = word[:-1]

    # Rule 2: Y to I
    elif word.endswith("y") and suffix in ["es", "ed"]:
        word = word[:-1] + "i"

    # Rule 3: Consonant doubling
    elif word == "stop" and suffix == "ed":
        word = "stopp"

    # Rule 4: Sibilant plural
    if suffix == "s" and word.endswith(
        ("s", "sh", "ch", "x", "z")
    ):
        suffix = "es"

    return word + suffix


print(apply_rules("diagnose", "ing"))
print(apply_rules("study", "es"))
print(apply_rules("stop", "ed"))
print(apply_rules("box", "s"))
print(apply_rules("care", "ful"))

diagnosing
studies
stopped
boxes
careful


Task 9

In [10]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words = [
    "infections",
    "fractures",
    "treated",
    "patients",
    "hospitals",
    "recovering"
]

print("Original Word\tStemmed Word")

for word in words:
    result = stemmer.stem(word)
    print(word, "\t\t", result)

Original Word	Stemmed Word
infections 		 infect
fractures 		 fractur
treated 		 treat
patients 		 patient
hospitals 		 hospit
recovering 		 recov


Task 10

In [11]:
import re
from nltk.stem import PorterStemmer

# Sample hospital record
text = """
Patient MRN123456 admitted on 05/11/2026.
Patient has fractures and infections.
Given 500mg Paracetamol BD.
BP is 120/80 mmHg.
"""

# Stage 1: Tokenisation
tokens = re.findall(
    r'\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?[A-Za-z]*'
    r'|[A-Za-z]+',
    text
)

print("1. Tokenisation:")
print(tokens)

# Stage 2: Entity Recognition
patterns = {
    "Patient ID": r'\bMRN[0-9]{6}\b',
    "Date": r'\b\d{2}/\d{2}/\d{4}\b',
    "Drug Strength": r'\b[A-Za-z]+\s+\d+mg\b',
    "Dosage": r'\b(?:OD|BD|TDS|QID)\b',
    "Blood Pressure": r'\b\d{2,3}/\d{2,3}\s?mmHg\b'
}

print("\n2. Entity Recognition:")

for name, pattern in patterns.items():
    print(name, ":", re.findall(pattern, text))

# Stage 3: Normalisation
normalised = text.lower()

normalised = re.sub(
    r'\b05/11/2026\b',
    '2026-11-05',
    normalised
)

normalised = re.sub(
    r'\bbd\b',
    'twice daily',
    normalised
)

print("\n3. Normalised Text:")
print(normalised)

# Stage 4: DFA Validation
def validate_id(patient_id):
    return bool(
        re.fullmatch(r'MRN[0-9]{6}', patient_id)
    )

print("\n4. DFA Validation:")
print(validate_id("MRN123456"))

# Stage 5: Morphological Analysis
lexicon = {
    "fractures": "fracture +N +PL",
    "infections": "infection +N +PL",
    "treated": "treat +V +PAST"
}

print("\n5. Morphological Analysis:")

for word, analysis in lexicon.items():
    print(word, "->", analysis)

# Stage 6: Porter Stemming
stemmer = PorterStemmer()

print("\n6. Porter Stemming:")

for word in ["fractures", "infections", "treated"]:
    print(word, "->", stemmer.stem(word))

1. Tokenisation:
['Patient', 'MRN', '123456', 'admitted', 'on', '05/11', '2026', 'Patient', 'has', 'fractures', 'and', 'infections', 'Given', '500mg', 'Paracetamol', 'BD', 'BP', 'is', '120/80', 'mmHg']

2. Entity Recognition:
Patient ID : ['MRN123456']
Date : ['05/11/2026']
Drug Strength : ['Given 500mg']
Dosage : ['BD']
Blood Pressure : ['120/80 mmHg']

3. Normalised Text:

patient mrn123456 admitted on 2026-11-05.
patient has fractures and infections.
given 500mg paracetamol twice daily.
bp is 120/80 mmhg.


4. DFA Validation:
True

5. Morphological Analysis:
fractures -> fracture +N +PL
infections -> infection +N +PL
treated -> treat +V +PAST

6. Porter Stemming:
fractures -> fractur
infections -> infect
treated -> treat


In [12]:
from collections import deque

nfa = {
    'q0': {'a': {'q0', 'q1'}},
    'q1': {'b': {'q2'}},
    'q2': {}
}

start = frozenset(['q0'])
final = {'q2'}

def move(states, symbol):
    result = set()
    for state in states:
        result.update(nfa.get(state, {}).get(symbol, set()))
    return frozenset(result)

def subset_construction():
    queue = deque([start])
    visited = {start}

    while queue:
        current = queue.popleft()

        for symbol in ['a', 'b']:
            next_state = move(current, symbol)

            if next_state and next_state not in visited:
                visited.add(next_state)
                queue.append(next_state)

    for state in visited:
        print(state, "Final:", bool(set(state) & final))

subset_construction()

frozenset({'q1', 'q0'}) Final: False
frozenset({'q0'}) Final: False
frozenset({'q2'}) Final: True


In [13]:
fst = {
    "fractures": "fracture +N +PL",
    "patients": "patient +N +PL",
    "treated": "treat +V +PAST"
}

def parse(word):
    return fst.get(word, "Not found")

def generate(analysis):
    for surface, lexical in fst.items():
        if lexical == analysis:
            return surface
    return "Not found"

print(parse("fractures"))
print(generate("fracture +N +PL"))

fracture +N +PL
fractures
